# EfficientMatch -- Expérience Fast FixMatch (FixMatch + Curriculum Batch Size)

Implémente **Fast FixMatch** (Chen, Dun & Kyrillidis, 2023/2024), limité au levier **Curriculum Batch Size (CBS)** -- le contributeur principal selon l'ablation des auteurs -- par-dessus FixMatch. Structure identique aux autres notebooks (sections 1, 3-5, 7).

**Différences avec FixMatch** : (1) la taille du batch non labellisé $u_t$ suit un schedule B-EXP croissant au lieu d'être fixe à $\mu B$ ; (2) le poids de la perte non supervisée $\lambda_u$ est recalé linéairement sur $u_t/B$ au lieu d'être fixe ; (3) les FLOPs par itération ne sont plus constants -- ils doivent être **sommés** itération par itération plutôt que multipliés par une constante.

## 1. Imports

In [ ]:
import os
import time
import json
import math
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.utils.flop_counter import FlopCounterMode
import torchvision
import torchvision.transforms as transforms_v1
import torchvision.transforms.v2 as transforms_v2

print(f"Torch version: {torch.__version__}, CUDA disponible: {torch.cuda.is_available()}")

## 2. Configuration

In [ ]:
CONFIG = {
    "dataset": "cifar10", "data_root": "./data", "n_labels": 40, "num_classes": 10,
    "B": 64, "mu": 7, "lr": 0.03, "momentum": 0.9, "nesterov": True, "weight_decay": 5e-4,
    "tau": 0.95, "lambda_u": 1.0, "ema_decay": 0.999,
    "K": 2 ** 17, "iters_per_epoch": 1024, "eval_every": 512, "seed": 0,
    "early_stopping": True, "es_window": 5, "es_slope_threshold": 1e-4,
    "use_amp": True, "cudnn_benchmark": True, "channels_last": True,
    "use_transforms_v2": True,  # True = torchvision.transforms.v2 (batch vectorisé) / False = v1 classique (par image)
    "num_workers": 4, "persistent_workers": True, "pin_memory": True,
    "debug_subset_size": None, "compile_model": False,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "log_path": "./logs_fast_fixmatch.json",
    "cbs_alpha": 0.7,       # sweet spot rapporte par les auteurs (Table 4 du papier)
    "cbs_min_batch": 8,     # borne basse pour eviter un batch quasi-vide en debut d'entrainement
}

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
if CONFIG["cudnn_benchmark"]:
    torch.backends.cudnn.benchmark = True
device = torch.device(CONFIG["device"])
print("Config chargée. Device:", device)

## 3. Traitement des données

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2471, 0.2435, 0.2616)

USE_TRANSFORMS_V2 = CONFIG["use_transforms_v2"]
T = transforms_v2 if USE_TRANSFORMS_V2 else transforms_v1

if USE_TRANSFORMS_V2:
    # v2 : un seul appel vectorisé sur tout le batch (CPU ou GPU), plus de boucle Python par image
    weak_transform = T.Compose([
        T.RandomHorizontalFlip(), T.RandomCrop(32, padding=4, padding_mode="reflect"),
        T.ToDtype(torch.float32, scale=True), T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])

    strong_transform = T.Compose([
        T.RandomHorizontalFlip(), T.RandomCrop(32, padding=4, padding_mode="reflect"),
        T.RandAugment(num_ops=2, magnitude=10),
        T.ToDtype(torch.float32, scale=True), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5),
    ])

    eval_transform = T.Compose([T.PILToTensor(), T.ToDtype(torch.float32, scale=True), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
else:
    # v1 (classique) : transform appliqué image par image (boucle Python) dans la boucle d'entraînement
    weak_transform = T.Compose([
        T.RandomHorizontalFlip(), T.RandomCrop(32, padding=4, padding_mode="reflect"),
        T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])

    strong_transform = T.Compose([
        T.RandomHorizontalFlip(), T.RandomCrop(32, padding=4, padding_mode="reflect"),
        T.RandAugment(num_ops=2, magnitude=10),
        T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5),
    ])

    eval_transform = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])


def apply_batch(transform, raw_batch):
    """Bascule transparente v1 (liste de PIL, boucle Python) / v2 (batch tenseur vectorisé)."""
    if USE_TRANSFORMS_V2:
        return transform(raw_batch.to(device, non_blocking=True))
    return torch.stack([transform(img) for img in raw_batch]).to(device, non_blocking=True)


def ssl_collate(batch):
    """Collate custom : le DataLoader ne sait pas empiler nativement une liste d'images PIL (mode v1)."""
    imgs, labels = zip(*batch)
    imgs = torch.stack(imgs) if USE_TRANSFORMS_V2 else list(imgs)
    return imgs, torch.tensor(labels)


class SSLDataset(Dataset):
    def __init__(self, base_dataset, indices):
        self.base_dataset = base_dataset
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, label = self.base_dataset[self.indices[idx]]
        if USE_TRANSFORMS_V2:
            img = transforms_v2.functional.pil_to_tensor(img)  # uint8 CHW -> collate direct en batch tenseur
        return img, label


def make_ssl_split(base_dataset, n_labels, num_classes, seed=0):
    rng = np.random.RandomState(seed)
    targets = np.array(base_dataset.targets)
    n_per_class = n_labels // num_classes
    labeled_idx = []
    for c in range(num_classes):
        idx_c = np.where(targets == c)[0]
        rng.shuffle(idx_c)
        labeled_idx.extend(idx_c[:n_per_class].tolist())
    labeled_idx = np.array(labeled_idx)
    unlabeled_idx = np.arange(len(base_dataset))
    return labeled_idx, unlabeled_idx


def load_datasets(cfg):
    if cfg["dataset"] == "cifar10":
        train_base = torchvision.datasets.CIFAR10(cfg["data_root"], train=True, download=True)
        test_base = torchvision.datasets.CIFAR10(cfg["data_root"], train=False, download=True, transform=eval_transform)
    elif cfg["dataset"] == "cifar100":
        train_base = torchvision.datasets.CIFAR100(cfg["data_root"], train=True, download=True)
        test_base = torchvision.datasets.CIFAR100(cfg["data_root"], train=False, download=True, transform=eval_transform)
    else:
        raise NotImplementedError(
            f"Dataset {cfg['dataset']} non branché ici -- ajouter le chargement MedMNIST (PathMNIST) via medmnist.PathMNIST"
        )
    labeled_idx, unlabeled_idx = make_ssl_split(train_base, cfg["n_labels"], cfg["num_classes"], seed=cfg["seed"])
    if cfg["debug_subset_size"] is not None:
        unlabeled_idx = unlabeled_idx[: cfg["debug_subset_size"]]
        test_base = Subset(test_base, list(range(min(len(test_base), cfg["debug_subset_size"]))))
    labeled_set = SSLDataset(train_base, labeled_idx)
    unlabeled_set = SSLDataset(train_base, unlabeled_idx)
    return labeled_set, unlabeled_set, test_base


def infinite_loader(dataset, batch_size, cfg, shuffle=True):
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        num_workers=cfg["num_workers"], pin_memory=cfg["pin_memory"],
        persistent_workers=cfg["persistent_workers"] and cfg["num_workers"] > 0, drop_last=True,
        collate_fn=ssl_collate,
    )
    while True:
        for batch in loader:
            yield batch

## 4. Modèle : WideResNet-28-2

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_planes, out_planes, stride, drop_rate=0.0):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.relu1 = nn.LeakyReLU(0.1, inplace=True)
        self.conv1 = nn.Conv2d(in_planes, out_planes, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes)
        self.relu2 = nn.LeakyReLU(0.1, inplace=True)
        self.conv2 = nn.Conv2d(out_planes, out_planes, 3, stride=1, padding=1, bias=False)
        self.drop_rate = drop_rate
        self.equal_io = in_planes == out_planes and stride == 1
        self.shortcut = None if self.equal_io else nn.Conv2d(in_planes, out_planes, 1, stride=stride, bias=False)

    def forward(self, x):
        out = self.relu1(self.bn1(x))
        shortcut = x if self.equal_io else self.shortcut(out)
        out = self.conv1(out)
        out = self.relu2(self.bn2(out))
        if self.drop_rate > 0:
            out = F.dropout(out, p=self.drop_rate, training=self.training)
        out = self.conv2(out)
        return out + shortcut


class WideResNet(nn.Module):
    def __init__(self, num_classes=10, depth=28, widen_factor=2, drop_rate=0.0):
        super().__init__()
        n_channels = [16, 16 * widen_factor, 32 * widen_factor, 64 * widen_factor]
        assert (depth - 4) % 6 == 0
        n = (depth - 4) // 6
        self.conv1 = nn.Conv2d(3, n_channels[0], 3, stride=1, padding=1, bias=False)
        self.block1 = self._make_block(n_channels[0], n_channels[1], n, stride=1, drop_rate=drop_rate)
        self.block2 = self._make_block(n_channels[1], n_channels[2], n, stride=2, drop_rate=drop_rate)
        self.block3 = self._make_block(n_channels[2], n_channels[3], n, stride=2, drop_rate=drop_rate)
        self.bn1 = nn.BatchNorm2d(n_channels[3])
        self.relu = nn.LeakyReLU(0.1, inplace=True)
        self.fc = nn.Linear(n_channels[3], num_classes)
        self.n_channels = n_channels[3]

    def _make_block(self, in_planes, out_planes, num_layers, stride, drop_rate):
        layers = [BasicBlock(in_planes, out_planes, stride, drop_rate)]
        for _ in range(1, num_layers):
            layers.append(BasicBlock(out_planes, out_planes, 1, drop_rate))
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.block1(out)
        out = self.block2(out)
        out = self.block3(out)
        out = self.relu(self.bn1(out))
        out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        return self.fc(out)


def build_model(cfg):
    model = WideResNet(num_classes=cfg["num_classes"], depth=28, widen_factor=2)
    model = model.to(device)
    if cfg["channels_last"]:
        model = model.to(memory_format=torch.channels_last)
    if cfg["compile_model"]:
        model = torch.compile(model)
    return model

## 5. EMA, FLOPs (mesure réelle) et évaluation

In [ ]:
class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)


def estimate_flops_for_batch_size(model, cfg, u_t):
    """Mesure les FLOPs (forward + backward) d'UNE itération pour une taille de batch non
    labellisé u_t donnée. Nécessaire car u_t varie avec k dans Fast FixMatch (CBS) -- les FLOPs
    par itération ne sont donc PAS constants ici, contrairement à FixMatch/FlexMatch/MixMatch.
    """
    device_ = next(model.parameters()).device
    model.train()
    B = cfg["B"]
    dummy_x = torch.randn(B, 3, 32, 32, device=device_)
    dummy_u_w = torch.randn(u_t, 3, 32, 32, device=device_)
    dummy_u_s = torch.randn(u_t, 3, 32, 32, device=device_)
    dummy_labels_x = torch.randint(0, cfg["num_classes"], (B,), device=device_)
    dummy_labels_u = torch.randint(0, cfg["num_classes"], (u_t,), device=device_)
    model.zero_grad(set_to_none=True)
    with FlopCounterMode(display=False) as flop_counter:
        logits_x = model(dummy_x)
        with torch.no_grad():
            _ = model(dummy_u_w)
        logits_u_s = model(dummy_u_s)
        loss = F.cross_entropy(logits_x, dummy_labels_x) + F.cross_entropy(logits_u_s, dummy_labels_u)
        loss.backward()
    model.zero_grad(set_to_none=True)
    return flop_counter.get_total_flops()


# Table de correspondance u_t -> FLOPs, pré-calculée à quelques points puis interpolée linéairement
# (éviter d'appeler FlopCounterMode à CHAQUE itération : trop lent, cf. discussion précédente).
def build_flops_lookup(model, cfg, n_points=12):
    u_max = cfg["mu"] * cfg["B"]
    sample_us = np.linspace(cfg["cbs_min_batch"], u_max, n_points, dtype=int)
    flops_samples = [estimate_flops_for_batch_size(model, cfg, int(u)) for u in sample_us]
    return sample_us, np.array(flops_samples)


def flops_from_lookup(u_t, sample_us, flops_samples):
    return float(np.interp(u_t, sample_us, flops_samples))


@torch.no_grad()
def evaluate(model, test_loader):
    model.eval()
    correct, total = 0, 0
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    model.train()
    return correct / total

## 6. Boucle d'entraînement -- Fast FixMatch (FixMatch + CBS)

Ajouts par rapport à `train_step_fixmatch` : calcul de $u_t$ (Étape 2bis) et recalage de $\lambda_u$ (Étape 7).

In [ ]:
def compute_curriculum_batch_size(k, cfg):
    """Curriculum Batch Size (CBS), formule B-EXP exacte de Chen, Dun & Kyrillidis (2023/2024) :
    u_t = u_max * (1 - (1 - t/T) / ((1-alpha) + alpha*(1 - t/T))), alpha = 0.7 (sweet spot,
    Table 4 du papier). Croissance lente puis accélérée, atteint u_max exactement à t=T.
    """
    u_max = cfg["mu"] * cfg["B"]
    t_ratio = k / cfg["K"]
    alpha = cfg["cbs_alpha"]
    u_t = u_max * (1 - (1 - t_ratio) / ((1 - alpha) + alpha * (1 - t_ratio)))
    u_t = max(int(u_t), cfg["cbs_min_batch"])
    return min(u_t, u_max)


def train_step_fast_fixmatch(model, ema, optimizer, scaler, k, cfg, labeled_iter, unlabeled_iter):
    """Une itération d'entraînement Fast FixMatch = FixMatch + Curriculum Batch Size.
    Comparer à `train_step_fixmatch` : Étapes 2 (troncature à u_t) et 7 (lambda_u recalé) sont
    les seuls ajouts -- tout le reste (perte supervisée, seuillage, backward, EMA) est identique.

    Le batch non labellisé est tiré à taille max (mu*B) via le DataLoader puis TRONQUE à u_t
    éléments -- plus simple que de reconfigurer dynamiquement le DataLoader, et équivalent en
    résultat puisque le batch est mélangé aléatoirement en amont.
    """
    # --- Étape 1 : batch labellisé (taille fixe l = B) ---
    imgs_x_raw, labels_x = next(labeled_iter)
    imgs_x = apply_batch(weak_transform, imgs_x_raw)
    labels_x = labels_x.to(device, non_blocking=True)

    # --- Étape 2 : CURRICULUM BATCH SIZE -- taille du batch non labellisé au temps k ---
    u_t = compute_curriculum_batch_size(k, cfg)
    imgs_u_raw_full, _ = next(unlabeled_iter)   # tiré a taille max (mu*B)
    imgs_u_raw = imgs_u_raw_full[:u_t]          # tronque a u_t elements pour cette iteration

    imgs_u_w = apply_batch(weak_transform, imgs_u_raw)
    imgs_u_s = apply_batch(strong_transform, imgs_u_raw)

    if cfg["channels_last"]:
        imgs_x = imgs_x.to(memory_format=torch.channels_last)
        imgs_u_w = imgs_u_w.to(memory_format=torch.channels_last)
        imgs_u_s = imgs_u_s.to(memory_format=torch.channels_last)

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(device_type=cfg["device"], enabled=cfg["use_amp"]):
        # --- Étape 3 : perte supervisée ---
        logits_x = model(imgs_x)
        loss_s = F.cross_entropy(logits_x, labels_x)

        # --- Étape 4 : pseudo-étiquetage sur la vue faible (sans gradient) ---
        with torch.no_grad():
            logits_u_w = model(imgs_u_w)
            probs_u_w = F.softmax(logits_u_w, dim=-1)
            max_probs, pseudo_labels = probs_u_w.max(dim=-1)

        # --- Étape 5 : seuil fixe (comme FixMatch -- CPL non inclus dans cette version) ---
        mask = max_probs.ge(cfg["tau"]).float()

        # --- Étape 6 : perte de cohérence faible/forte, filtrée par le masque ---
        logits_u_s = model(imgs_u_s)
        loss_u_per_sample = F.cross_entropy(logits_u_s, pseudo_labels, reduction="none")
        loss_u = (loss_u_per_sample * mask).mean()

        # --- Étape 7 : perte totale, lambda_u recalé sur le ratio de taille de batch courant ---
        lambda_u_t = u_t / cfg["B"]
        loss = loss_s + lambda_u_t * loss_u

    # --- Étape 8 : backward + optimisation ---
    if cfg["use_amp"]:
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        optimizer.step()

    # --- Étape 9 : mise à jour EMA ---
    ema.update(model)

    return {
        "loss": loss.item(), "loss_s": loss_s.item(), "loss_u": loss_u.item(),
        "mask_rate": mask.mean().item(), "u_t": u_t, "lambda_u_t": lambda_u_t,
    }

## 7. Assemblage : modèle, optimiseur, scheduler

In [ ]:
def cosine_schedule(optimizer, k, K):
    base_lr = optimizer.defaults["lr"]
    new_lr = base_lr * math.cos(7 * math.pi * k / (16 * K))
    for group in optimizer.param_groups:
        group["lr"] = max(new_lr, 0.0)


labeled_set, unlabeled_set, test_set = load_datasets(CONFIG)
labeled_iter = infinite_loader(labeled_set, CONFIG["B"], CONFIG, shuffle=True)
unlabeled_iter = infinite_loader(unlabeled_set, CONFIG["mu"] * CONFIG["B"], CONFIG, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=CONFIG["num_workers"])

model = build_model(CONFIG)
ema = EMA(model, CONFIG["ema_decay"])
optimizer = torch.optim.SGD(model.parameters(), lr=CONFIG["lr"], momentum=CONFIG["momentum"],
                             nesterov=CONFIG["nesterov"], weight_decay=CONFIG["weight_decay"])
scaler = torch.amp.GradScaler(enabled=CONFIG["use_amp"])

# Mesure UNIQUE avant l'entraînement : table u_t -> FLOPs, interpolée ensuite pendant la boucle
sample_us, flops_samples = build_flops_lookup(model, CONFIG)
print("Table FLOPs(u_t) construite sur", len(sample_us), "points.")
print(f"Budget total : {CONFIG['K']} itérations")

In [ ]:
def detect_plateau(acc_history, window=5, slope_threshold=1e-4):
    """Détecte un plateau via la pente d'une régression linéaire locale sur l'accuracy EMA.
    Préféré à un compteur de patience car la pente s'adapte à l'échelle locale du bruit,
    plutôt que de dépendre d'un seuil absolu sur l'accuracy (cf. justification papier :
    argument tiré de la décroissance du schedule cosine, qui porte sur un TAUX de variation).
    """
    if len(acc_history) < window:
        return False, None
    recent = np.array(acc_history[-window:])
    x = np.arange(window)
    slope = np.polyfit(x, recent, 1)[0]
    return abs(slope) < slope_threshold, slope

## 8. Boucle principale + logging

In [ ]:
logs = []
acc_history = []
eval_model = build_model(CONFIG)
start_time = time.time()
cumulative_flops = 0.0  # FLOPs NON constants ici -- on les SOMME itération par itération
model.train()

for k in range(1, CONFIG["K"] + 1):
    cosine_schedule(optimizer, k, CONFIG["K"])
    step_metrics = train_step_fast_fixmatch(model, ema, optimizer, scaler, k, CONFIG,
                                             labeled_iter, unlabeled_iter)
    cumulative_flops += flops_from_lookup(step_metrics["u_t"], sample_us, flops_samples)

    if k % CONFIG["eval_every"] == 0 or k == CONFIG["K"]:
        ema.copy_to(eval_model)
        acc = evaluate(eval_model, test_loader)
        elapsed = time.time() - start_time
        log_entry = {"iteration": k, "elapsed_seconds": elapsed, "cumulative_flops": cumulative_flops,
                     "eval_accuracy": acc, **step_metrics}
        logs.append(log_entry)
        print(f"[iter {k:>7}/{CONFIG['K']}] acc={acc:.4f} loss={step_metrics['loss']:.4f} "
              f"u_t={step_metrics['u_t']} mask_rate={step_metrics['mask_rate']:.3f} "
              f"elapsed={elapsed/60:.1f}min")
        with open(CONFIG["log_path"], "w") as f:
            json.dump({"config": CONFIG, "logs": logs}, f, indent=2)

        acc_history.append(acc)
        if CONFIG["early_stopping"]:
            is_plateau, slope = detect_plateau(acc_history, CONFIG["es_window"], CONFIG["es_slope_threshold"])
            if is_plateau:
                print(f"Plateau détecté (pente={slope:.2e} < seuil={CONFIG['es_slope_threshold']:.2e}) "
                      f"-- arrêt anticipé à l'itération {k}.")
                break

print("Entraînement terminé.")